# 🏆 Kaggle: Name That LLM — Frontier Model Attribution
### Adaptive Family Gates + Deep Stylometry + Sinkhorn-Knopp Balancing
---
- **Task**: Fine-Grained Authorship Attribution across 18 Frontier LLMs
- **Evaluation Metric**: Balanced Accuracy
- **Target Submission**: `submission.parquet` [columns: `id`, `model`]
- **Key Insights**:
  1. **Strict Grouped CV**: StratifiedGroupKFold on `prompt_id` to prevent topic leakage.
  2. **120+ Stylometric Signals**: Capturing punctuation (em-dashes, semicolons, Oxford commas), markdown formatting (bullets, headers), lexical diversity (TTR, Hapax, Yule's K), and discourse markers.
  3. **Adaptive Family Gates**: Hierarchical classification into 6 AI lab families, then intra-family attribution.
  4. **Sinkhorn-Knopp Prior Balancing**: Optimal transport marginal balancing to maximize Balanced Accuracy without falling into the brittle Hungarian 1-to-1 matching trap.


In [ ]:
# Core Imports
import os, sys, re, math, collections, gc
from pathlib import Path
from typing import Dict, List, Optional, Tuple, Union
import numpy as np
import pandas as pd
import scipy.sparse as sp
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import RidgeClassifier, LogisticRegression
from sklearn.metrics import balanced_accuracy_score, recall_score, confusion_matrix
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler

print('[+] Libraries successfully imported.')


## 1. Candidate Models & Family Taxonomy
We define the 18 candidate frontier LLM classes and map them to their parent AI lab families.

In [ ]:
CANDIDATE_MODELS = [
    'gpt-5.5', 'gpt-5.6-luna', 'gpt-5.6-sol', 'gpt-5.6-terra', 'gpt-6-astra',
    'gemini-3.1-pro-preview', 'gemini-3.6-flash', 'gemini-3.7-flash', 'gemini-3.8-flash',
    'claude-fable-5', 'claude-opus-4-6', 'claude-opus-4-8', 'claude-opus-5', 'claude-sonnet-5',
    'qwen3.8-max', 'glm-5.3', 'glm-5.3-flash', 'kimi-k3'
]
NUM_CLASSES = len(CANDIDATE_MODELS)
MODEL_TO_IDX = {m: i for i, m in enumerate(CANDIDATE_MODELS)}
IDX_TO_MODEL = {i: m for i, m in enumerate(CANDIDATE_MODELS)}

MODEL_FAMILY_MAP = {
    'gpt-5.5': 'OpenAI', 'gpt-5.6-luna': 'OpenAI', 'gpt-5.6-sol': 'OpenAI', 'gpt-5.6-terra': 'OpenAI', 'gpt-6-astra': 'OpenAI',
    'gemini-3.1-pro-preview': 'Google', 'gemini-3.6-flash': 'Google', 'gemini-3.7-flash': 'Google', 'gemini-3.8-flash': 'Google',
    'claude-fable-5': 'Anthropic', 'claude-opus-4-6': 'Anthropic', 'claude-opus-4-8': 'Anthropic', 'claude-opus-5': 'Anthropic', 'claude-sonnet-5': 'Anthropic',
    'qwen3.8-max': 'Qwen', 'glm-5.3': 'ZAI', 'glm-5.3-flash': 'ZAI', 'kimi-k3': 'Moonshot'
}
FAMILY_NAMES = ['OpenAI', 'Anthropic', 'Google', 'ZAI', 'Qwen', 'Moonshot']
FAMILY_TO_IDX = {f: i for i, f in enumerate(FAMILY_NAMES)}
FAMILY_TO_MODEL_INDICES = {f: [MODEL_TO_IDX[m] for m, fam in MODEL_FAMILY_MAP.items() if fam == f] for f in FAMILY_NAMES}
print(f'[+] Configured {NUM_CLASSES} models across {len(FAMILY_NAMES)} families.')


## 2. Dataset Auto-Discovery
Searches for `train.csv` and `test.csv` in Kaggle competition paths.

In [ ]:
def locate_file(filename: str) -> Optional[Path]:
    search_dirs = [Path('/kaggle/input/name-that-llm'), Path('/kaggle/input/competitions/name-that-llm'), Path('./data'), Path('.')]
    for d in search_dirs:
        candidate = d / filename
        if candidate.exists(): return candidate
    kaggle_in = Path('/kaggle/input')
    if kaggle_in.exists():
        found = list(kaggle_in.glob(f'**/{filename}'))
        if found: return found[0]
    return None

train_path = locate_file('train.csv')
test_path = locate_file('test.csv')

if train_path and train_path.exists():
    train_df = pd.read_csv(train_path)
    print(f'[*] Loaded train.csv: {train_df.shape}')
else:
    print('[!] train.csv not found; generating mock validation dataset.')
    from src.synthetic_data_generator import generate_synthetic_dataset
    train_df, test_df = generate_synthetic_dataset(n_prompts=15)

if test_path and test_path.exists():
    test_df = pd.read_csv(test_path)
print(f'[*] Loaded test.csv: {test_df.shape}')


## 3. Stylometric & Dual TF-IDF Feature Extraction
Computes 120+ stylometric markers combined with sublinear word and character n-grams.

In [ ]:
from src.stylometry import StylometryExtractor
from src.tfidf_features import DualTfidfExtractor

print('[*] Extracting stylometric signals...')
sty_extractor = StylometryExtractor()
X_sty_train = sty_extractor.fit_transform(train_df['text'])
X_sty_test = sty_extractor.transform(test_df['text'])

scaler = StandardScaler()
X_sty_train_s = sp.csr_matrix(scaler.fit_transform(X_sty_train))
X_sty_test_s = sp.csr_matrix(scaler.transform(X_sty_test))

print('[*] Extracting dual sublinear TF-IDF...')
tfidf_extractor = DualTfidfExtractor()
X_tfidf_train = tfidf_extractor.fit_transform(list(train_df['text']))
X_tfidf_test = tfidf_extractor.transform(list(test_df['text']))

X_train = sp.hstack([X_sty_train_s, X_tfidf_train], format='csr')
X_test = sp.hstack([X_sty_test_s, X_tfidf_test], format='csr')
print(f'[+] Combined feature space dimension: {X_train.shape[1]}')


## 4. Grouped Cross-Validation & Adaptive Family Gates Model
Strict `StratifiedGroupKFold` on `prompt_id` to ensure zero topic leakage across train and validation folds.

In [ ]:
from src.models_hierarchical import AdaptiveFamilyGateClassifier
from src.postprocessing import sinkhorn_knopp_balancing

y_train = np.array([MODEL_TO_IDX[m] for m in train_df['model']])
groups = train_df['prompt_id'].values if 'prompt_id' in train_df.columns else np.arange(len(train_df))
n_splits = 5 if len(np.unique(groups)) >= 5 else 2
sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=42)

models = []
oof_probs = np.zeros((len(train_df), NUM_CLASSES), dtype=np.float32)

for fold, (trn_idx, val_idx) in enumerate(sgkf.split(np.zeros(len(train_df)), y_train, groups=groups)):
    X_trn, y_trn = X_train[trn_idx], y_train[trn_idx]
    X_val, y_val = X_train[val_idx], y_train[val_idx]
    
    model = AdaptiveFamilyGateClassifier(alpha_family=1.0, alpha_model=0.8, hierarchical_weight=0.4)
    model.fit(X_trn, y_trn)
    models.append(model)
    
    val_probs = model.predict_proba(X_val)
    oof_probs[val_idx] = val_probs
    val_acc = balanced_accuracy_score(y_val, np.argmax(val_probs, axis=1))
    print(f'  -> Fold {fold + 1}/{n_splits} Balanced Accuracy: {val_acc:.4f}')

raw_oof_acc = balanced_accuracy_score(y_train, np.argmax(oof_probs, axis=1))
print(f'\n[*] Raw Out-of-Fold Balanced Accuracy: {raw_oof_acc:.4f}')

calibrated_oof = sinkhorn_knopp_balancing(oof_probs)
calib_oof_acc = balanced_accuracy_score(y_train, np.argmax(calibrated_oof, axis=1))
print(f'[*] Sinkhorn-Calibrated Out-of-Fold Balanced Accuracy: {calib_oof_acc:.4f}')


## 5. Test Inference & Parquet Export
Generate ensemble predictions, apply Sinkhorn marginal balancing, and save to `submission.parquet`.

In [ ]:
print(f'[*] Predicting on {len(test_df)} test passages...')
test_probs = np.zeros((len(test_df), NUM_CLASSES), dtype=np.float32)
for model in models:
    test_probs += model.predict_proba(X_test) / len(models)

# Sinkhorn optimal transport marginal balancing
test_probs_balanced = sinkhorn_knopp_balancing(test_probs)
pred_indices = np.argmax(test_probs_balanced, axis=1)
pred_models = [IDX_TO_MODEL[i] for i in pred_indices]

sub_df = pd.DataFrame({
    'id': test_df['id'].values,
    'model': pred_models
})

sub_df.to_parquet('submission.parquet', index=False)
print(f'[+] submission.parquet successfully created: {sub_df.shape}')
print(sub_df.head(10))
